# Force distribution and natural modes of a truss bridge

This notebook studies the mechanical behaviour of a bridge modelled as a plane
truss. Using the direct stiffness method, it computes how forces distribute
through the structure under load, simulates the passage of a vehicle together
with a simple failure criterion, adds the bars' own weight, and finally
determines the natural modes of vibration.

It was written as a course project, to connect the finite-element formulation
seen in lectures with a concrete implementation, and is shared so that it may be
read and reused. The method is implemented in the accompanying module
`truss.py`; this notebook applies it to one particular bridge and discusses the
results.

## 1. The model

A truss is a set of straight bars joined at frictionless pins. Each bar carries
only an axial force, in tension or compression, and bending is neglected. Every
node has two degrees of freedom, its horizontal and vertical displacement, so a
truss with $N$ nodes has $2N$ of them, and the static problem is the linear
system

$$ K\,u = F , $$

where $K$ is the global stiffness matrix, $u$ the nodal displacements, and $F$
the applied nodal forces.

For a single bar of length $L$, section $A$ and Young's modulus $E$, making an
angle $\theta$ with the horizontal, the stiffness in global coordinates is
obtained by projecting the axial stiffness $EA/L$ onto the axes with
$c = \cos\theta$ and $s = \sin\theta$:

$$ K_e = \frac{EA}{L} \begin{pmatrix} c^2 & cs & -c^2 & -cs \\ cs & s^2 & -cs & -s^2 \\ -c^2 & -cs & c^2 & cs \\ -cs & -s^2 & cs & s^2 \end{pmatrix} . $$

The global matrix $K$ is assembled by adding each bar's contribution at the
degrees of freedom of the nodes it connects. The supports are imposed by holding
the corresponding displacements at zero, which amounts to removing their rows
and columns before solving.

The bridge studied here has seven nodes, four along the deck and three above,
joined into equilateral triangles of side $10\,\mathrm{m}$. The end nodes N1 and
N4 are pinned to the banks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable

import truss

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
UNDEFORMED = "#c3c7cc"
NODE = "#22303f"
LOAD = "#e76f51"

In [ ]:
# Geometry of the bridge.
height = 5 * np.sqrt(3)
nodes = np.array([
    [0.0, 0.0], [10.0, 0.0], [20.0, 0.0], [30.0, 0.0],
    [5.0, height], [15.0, height], [25.0, height],
])

# Bars, as pairs of node indices (zero-based). The first three are the deck.
bars = [(0, 1), (1, 2), (2, 3), (3, 6), (5, 6), (4, 5),
        (0, 4), (1, 4), (1, 5), (2, 5), (2, 6)]

deck_nodes = [0, 1, 2, 3]
fixed_dofs = [0, 1, 6, 7]              # N1 and N4 pinned: their x and y are held at 0

E = 2.0e11                             # Young's modulus, N/m^2
A = 0.01                              # cross-section, m^2
linear_density = 7850 * A             # steel bar, kg/m

K = truss.assemble_stiffness(nodes, bars, E, A)
print("stiffness matrix:", K.shape)
print("structure is stable:", truss.is_stable(K, fixed_dofs))

In [ ]:
# Helpers used throughout to display the truss.

def deformed(displacement, amplify):
    # Node coordinates after adding the (amplified) displacements.
    return nodes + amplify * displacement.reshape(-1, 2)


def auto_amplify(displacement, target=4.0):
    # Scale factor bringing the largest displacement to `target` metres.
    largest = np.abs(displacement).max()
    return target / largest if largest > 0 else 1.0


def plot_forces(displacement, title):
    # Undeformed truss in grey, deformed truss coloured by axial force.
    axial = truss.all_axial_forces(nodes, bars, displacement, E, A) / 1000
    amplify = auto_amplify(displacement)
    points = deformed(displacement, amplify)
    limit = np.abs(axial).max()
    norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
    colormap = plt.get_cmap("coolwarm")

    fig, ax = plt.subplots(figsize=(8, 4.2))
    for i, j in bars:
        ax.plot(*zip(nodes[i], nodes[j]), color=UNDEFORMED, lw=1.5, zorder=1)
    for value, (i, j) in zip(axial, bars):
        ax.plot(*zip(points[i], points[j]), color=colormap(norm(value)), lw=3, zorder=2)
    ax.scatter(points[:, 0], points[:, 1], color=NODE, s=25, zorder=3)

    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_title(f"{title} (deformation ×{amplify:.0e})")
    bar = fig.colorbar(ScalarMappable(norm=norm, cmap=colormap), ax=ax, shrink=0.8)
    bar.set_label("axial force (kN)   — compression   + tension")
    plt.show()

## 2. Static loading

Two downward forces of $1000\,\mathrm{N}$ are applied at the deck nodes N2 and
N3. Solving $K u = F$ gives the nodal displacements, from which the deformed
shape and the axial force in every bar follow.

In [ ]:
forces = np.zeros(2 * len(nodes))
forces[2 * 1 + 1] = -1000            # vertical load at N2
forces[2 * 2 + 1] = -1000            # vertical load at N3

displacement = truss.solve_displacements(K, forces, fixed_dofs)
reactions = truss.support_reactions(K, displacement, forces)

print(f"reaction at N1: ({reactions[0]:7.1f}, {reactions[1]:7.1f}) N")
print(f"reaction at N4: ({reactions[6]:7.1f}, {reactions[7]:7.1f}) N")
print(f"sum of vertical reactions: {reactions[1] + reactions[7]:.1f} N  (applied load: 2000 N)")

plot_forces(displacement, "Static load on N2 and N3")

The vertical reactions at the two supports sum to the applied load of
$2000\,\mathrm{N}$, and by symmetry each support carries half. The horizontal
reactions are equal and opposite: they balance the inward thrust of the inclined
bars. The top chord works in compression and the loaded diagonals in tension, as
expected for this loading.

## 3. A vehicle crossing the bridge

The vehicle is modelled as a point load equal to its weight, moving along the
deck. Between two deck nodes the load is shared between them in proportion to its
position, so that a continuous crossing can be simulated. At each position the
system is solved and the largest axial force in the structure is recorded.

A bar is taken to fail when the axial force at one of its nodes exceeds an
admissible value. With the diagonals in place, the largest force during the
crossing stays below this threshold and the vehicle passes safely.

In [ ]:
vehicle_weight = 1000 * 9.81
admissible_force = 8000

def moving_load(x_car, weight):
    # Point load at position x_car, shared between the two nearest deck nodes.
    load = np.zeros(2 * len(nodes))
    segment = min(int(x_car // 10), len(deck_nodes) - 2)
    t = (x_car - 10 * segment) / 10
    load[2 * deck_nodes[segment] + 1] -= weight * (1 - t)
    load[2 * deck_nodes[segment + 1] + 1] -= weight * t
    return load


positions = np.linspace(0, 30, 200)
peak_force = []
for x_car in positions:
    u = truss.solve_displacements(K, moving_load(x_car, vehicle_weight), fixed_dofs)
    axial = np.abs(truss.all_axial_forces(nodes, bars, u, E, A))
    peak_force.append(axial.max())
peak_force = np.array(peak_force)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(positions, peak_force, color=LOAD, lw=2, label="largest axial force")
ax.axhline(admissible_force, color=NODE, ls="--", lw=1, label="admissible force")
ax.set_xlabel("vehicle position along the deck (m)")
ax.set_ylabel("axial force (N)")
ax.set_ylim(0, 1.1 * admissible_force)
ax.set_title("Largest axial force as the vehicle crosses")
ax.legend()
plt.show()

print(f"peak axial force during the crossing: {peak_force.max():.0f} N")
print(f"admissible force: {admissible_force} N  ->  the bridge holds")

### The role of the diagonals

Removing the diagonal bars leaves only the deck, the top chord, and the two end
bars. The reduced stiffness matrix is then singular: the structure is a
mechanism, able to deform without any bar stretching, and $K u = F$ no longer has
a unique solution. This is a physical result rather than a numerical error, and
it shows why the diagonals are essential.

In [ ]:
bars_without_diagonals = [(0, 1), (1, 2), (2, 3), (3, 6), (5, 6), (4, 5), (0, 4)]
K_without = truss.assemble_stiffness(nodes, bars_without_diagonals, E, A)

free = truss.free_dofs(K_without.shape[0], fixed_dofs)
reduced = K_without[np.ix_(free, free)]
print(f"stable: {truss.is_stable(K_without, fixed_dofs)}")
print(f"rank of reduced matrix: {np.linalg.matrix_rank(reduced)} out of {reduced.shape[0]}")

## 4. Adding the weight of the bars

The bridge also carries its own weight. Each bar's weight is split equally
between its two end nodes and added to the load. Because the internal forces then
grow, the deformation is markedly larger than under the external load alone.

In [ ]:
weight_forces = truss.self_weight_forces(nodes, bars, linear_density)
displacement = truss.solve_displacements(K, weight_forces, fixed_dofs)
plot_forces(displacement, "Under the self-weight of the bars")

## 5. Natural modes of vibration

Beyond its static response, the bridge has natural frequencies and associated
shapes of vibration, obtained from the generalised eigenvalue problem

$$ K\,\phi = \omega^2 M\,\phi , $$

where $M$ is a mass matrix. A lumped mass matrix is used: half of each bar's
mass is placed at each of its end nodes. The frequencies are $f = \omega / 2\pi$,
and the same supports as before are applied.

In [ ]:
M = truss.lumped_mass_matrix(nodes, bars, linear_density)
frequencies, shapes = truss.natural_modes(K, M, fixed_dofs, 6)

for mode, f in enumerate(frequencies):
    print(f"mode {mode + 1}: f = {f:.3f} Hz")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), constrained_layout=True)
for mode, ax in enumerate(axes.flat):
    points = deformed(shapes[:, mode], auto_amplify(shapes[:, mode], target=3.5))
    for i, j in bars:
        ax.plot(*zip(nodes[i], nodes[j]), color=UNDEFORMED, lw=1.2, zorder=1)
        ax.plot(*zip(points[i], points[j]), color=LOAD, lw=2.2, zorder=2)
    ax.set_aspect("equal")
    ax.set_xlim(-3, 33)
    ax.set_ylim(-6, 15)
    ax.set_title(f"mode {mode + 1}: f = {frequencies[mode]:.1f} Hz", fontsize=10)
    ax.grid(alpha=0.15)
fig.suptitle("First six natural modes of vibration")
plt.show()

The lowest modes are global bending shapes of the whole span; as the frequency
increases the shapes become more oscillatory and more localised. No
near-zero frequency appears, which is consistent with a properly constrained,
rigid structure.

## 6. Further directions

The same tools extend naturally to richer problems: a distributed or dynamic
vehicle load rather than a moving point force, a failure criterion based on the
stress in each bar or on buckling rather than a nodal force, or the inclusion of
structural damping. Other bridge geometries can be studied simply by changing the
lists of nodes and bars.

---
*Based on the direct stiffness method for plane trusses, a standard technique of
structural finite-element analysis.*